# Functional tests

Below is a list of functional tests which should be included in sending and receiving implementations.

### Sending

- Ensure taproot outputs are excluded during coin selection if the sender does not have access to the key path private key (unless using H as the taproot internal key)
- Ensure the silent payment address is re-derived if inputs are added or removed during RBF

### Receiving

- Ensure the public key can be extracted from non-standard P2PKH scriptSigs
- Ensure taproot script path spends are included, using the taproot output key (unless H is used as the taproot internal key)
- Ensure the scanner can extract the public key from each of the input types supported (e.g. P2WPKH, P2SH-P2WPKH, etc.)

In [ ]:
"""Run every pinned reference subcase using the same exact comparator as API/CI."""
import json
from pathlib import Path
import sys

api_dir = next(
    directory if (directory / "core/test/test_vectors.json").is_file() else directory / "api"
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "core/test/test_vectors.json").is_file()
    or (directory / "api/core/test/test_vectors.json").is_file()
)
sys.path.insert(0, str(api_dir))
from core.reference import run_case

vectors = json.loads((api_dir / "core/test/test_vectors.json").read_text())
results = []
for test_id, entry in enumerate(vectors):
    for direction in ("sending", "receiving"):
        for case_index in range(len(entry.get(direction, []))):
            result = run_case(entry, direction, case_index, test_id)
            results.append(result["test_passed"])
            print(f"{test_id}/{direction}/{case_index}: {'PASS' if result['test_passed'] else 'FAIL'}")
assert all(results), "Reference comparison failed"
print(f"All {len(results)} reference subcases passed.")
